# 04 · Strategy Research
#
**Question:** Can simple patterns in the odds be turned into profitable rules?
#
We backtest the predefined odds-only strategies with fixed £1 stakes over the
*full* primary dataset. **These are exploratory/in-sample results.** They must
not be taken as evidence of a real edge until confirmed out-of-sample (notebook 05).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd

from src import data as D
from src import probabilities as P
from src import strategies as S
from src import backtest as B
from src import metrics as M
from src import plotting as plt

primary = P.add_probability_columns(D.load_processed())


## Strategy definitions

In [2]:
for name, spec in S.STRATEGIES.items():
    print(f"* **{name}**: {spec['desc']}")


* **favourite**: Always bet the favourite (highest normalised-probability outcome).
* **favourite_strong**: Bet the favourite only when odds <= 2.0 (strong favourite).
* **prob_bucket_45**: Bet the favourite only when its normalised probability >= 0.45.
* **low_overround**: Bet the favourite only when book overround <= 0.10.
* **draw_value_30**: Bet the draw when its normalised probability >= 0.30.


## Backtest each strategy (in-sample)

In [3]:
results = {}
ledgers = {}
for name in S.STRATEGIES:
    ledger = B.backtest_strategy(primary, name)
    ledgers[name] = ledger
    m = M.summarize_ledger(ledger)
    m.update({"bet": False, "label": name})
    results[name] = m

overall = pd.DataFrame(results).T[
    ["label", "bets", "wins", "losses", "win_rate", "total_stake",
     "gross_return", "net_profit", "roi", "average_odds",
     "max_drawdown", "longest_losing_streak"]
].round(4)
overall


,label,bets,wins,losses,win_rate,total_stake,gross_return,net_profit,roi,average_odds,max_drawdown,longest_losing_streak
favourite,favourite,2280,1235,1045,0.541667,2280.0,2273.48,-6.52,-0.00286,1.944504,35.39,10
favourite_strong,favourite_strong,1272,804,468,0.632075,1272.0,1269.04,-2.96,-0.002327,1.619104,19.13,5
prob_bucket_45,prob_bucket_45,1425,878,547,0.61614,1425.0,1422.04,-2.96,-0.002077,1.667474,23.08,6
low_overround,low_overround,2277,1234,1043,0.541941,2277.0,2271.48,-5.52,-0.002424,1.94473,35.39,10
draw_value_30,draw_value_30,529,173,356,0.327032,529.0,519.12,-9.88,-0.018677,2.989735,40.49,10


## Cumulative P&L and drawdown for each strategy

In [4]:
for name in S.STRATEGIES:
    print(f"\n### {name}")
    fig = plt.cumulative_pnl(ledgers[name])
    fig.show()
    fig = plt.drawdown_curve(ledgers[name])
    fig.show()



### favourite



### favourite_strong



### prob_bucket_45



### low_overround



### draw_value_30


## Season-by-season ROI

In [5]:
for name in S.STRATEGIES:
    stbl = M.season_summary(ledgers[name])
    print(f"\n### {name}")
    print(stbl[["season", "bets", "win_rate", "net_profit", "roi"]].round(4).to_string())
    fig = plt.season_roi(stbl)
    fig.show()



### favourite
  season  bets  win_rate  net_profit     roi
0  20/21   380    0.5395       -7.29 -0.0192
1  21/22   380    0.5211      -15.85 -0.0417
2  22/23   380    0.5421        5.16  0.0136
3  23/24   380    0.5500        3.84  0.0101
4  24/25   380    0.5605        6.91  0.0182
5  25/26   380    0.5368        0.71  0.0019



### favourite_strong
  season  bets  win_rate  net_profit     roi
0  20/21   222    0.6171       -6.59 -0.0297
1  21/22   209    0.6268       -0.03 -0.0001
2  22/23   204    0.6176       -4.76 -0.0233
3  23/24   217    0.6452        6.33  0.0292
4  24/25   218    0.6468        0.20  0.0009
5  25/26   202    0.6386        1.89  0.0094



### prob_bucket_45
  season  bets  win_rate  net_profit     roi
0  20/21   252    0.6071       -3.49 -0.0138
1  21/22   231    0.6017       -5.53 -0.0239
2  22/23   234    0.6068       -1.71 -0.0073
3  23/24   240    0.6292        6.08  0.0253
4  24/25   243    0.6214       -4.15 -0.0171
5  25/26   225    0.6311        5.84  0.0260



### low_overround
  season  bets  win_rate  net_profit     roi
0  20/21   380    0.5395       -7.29 -0.0192
1  21/22   380    0.5211      -15.85 -0.0417
2  22/23   380    0.5421        5.16  0.0136
3  23/24   380    0.5500        3.84  0.0101
4  24/25   380    0.5605        6.91  0.0182
5  25/26   377    0.5385        1.71  0.0045



### draw_value_30
  season  bets  win_rate  net_profit     roi
0  20/21    85    0.3765       10.31  0.1213
1  21/22    84    0.3810       12.35  0.1470
2  22/23    84    0.2619      -17.76 -0.2114
3  23/24    78    0.3333        0.28  0.0036
4  24/25   113    0.3274       -2.40 -0.0212
5  25/26    85    0.2824      -12.66 -0.1489


## Key observations (in-sample)
#
* The `favourite` family produces many bets, a win rate near the favourite
  calibration (~54%), and an ROI close to but slightly below zero — i.e. the
  book's ~5.6% overround largely erases any favourite value.
* `draw_value_30` has a much lower win rate (~33%) and clearly negative ROI.
* None of these in-sample results, by themselves, demonstrate a persistent
  edge. The decisive test is on unseen data (next notebook).